In [31]:
from hta.trace_analysis import TraceAnalysis
import polars as pl
import glob

Couldn't parse gemma train.

In [32]:
bw_dfs = []
for path in glob.glob("/proj/threadtune-PG0/amir/kineto_traces/*"):
    analyzer = TraceAnalysis({0: "device.0.json"}, path)
    rank = 0
    trace_data = analyzer.t.get_trace(rank)
    symbol_table = analyzer.t.symbol_table.get_sym_table()
    bw_dfs.append(
        pl.concat(
            pl.from_pandas(trace_data[trace_data["stream"] != -1])
            .select(pl.lit(rank).alias("rank"), pl.all())
            for rank, trace_data in analyzer.t.traces.items()
        ).join(
            pl.from_dict({"name": list(range(len(symbol_table))), "s_name": symbol_table}),
            on="name",
        ).select(
            pl.lit(path).str.extract(r"/([^/]*)$").alias("model"),
            (pl.col("memory_bw_gbps") * pl.col("dur") * 1e-6).sum().alias("memory_use_gb"),
            ((pl.col("ts") + pl.col("dur")) * 1e-6).max().alias("time_s"),
        ).with_columns(
            (pl.col("memory_use_gb") / pl.col("time_s")).alias("memory_bw_gbps")
        )
    )
pl.concat(bw_dfs)

Parsed /proj/threadtune-PG0/amir/kineto_traces/deit-train/device.0.json time = 0.49 seconds 
Rounding down ns resolution events due to issue with events overlapping. ts dtype = float64, dur dtype = float64.Please see https://github.com/pytorch/pytorch/pull/122425
Parsed /proj/threadtune-PG0/amir/kineto_traces/deit-train/device.0.json backend=json in 0.77 seconds; current PID:142801
Overall parsing of /proj/threadtune-PG0/amir/kineto_traces/deit-train/device.0.json in 0.83 seconds; current PID:142801
leaving parse_multiple_ranks duration=0.84 seconds
leaving parse_traces duration=0.84 seconds
There is only one iteration in the trace. The analysis result may not be accurate.
Parsed /proj/threadtune-PG0/amir/kineto_traces/gpt2-inference/device.0.json time = 0.04 seconds 
Rounding down ns resolution events due to issue with events overlapping. ts dtype = float64, dur dtype = float64.Please see https://github.com/pytorch/pytorch/pull/122425
Parsed /proj/threadtune-PG0/amir/kineto_traces/gpt

model,memory_use_gb,time_s,memory_bw_gbps
str,f64,f64,f64
"""deit-train""",0.000662,0.665412,0.000996
"""gpt2-inference""",0.000352,0.464135,0.000758
"""bloom-train""",0.000001,0.436817,0.000003
"""resnet50-train""",0.011859,0.701243,0.016911
"""gpt2-train""",0.000001,0.581234,0.000002
…,…,…,…
"""bert-inference""",6.6489e-10,0.313939,2.1179e-9
"""deit-inference""",0.000011,0.322152,0.000035
"""bloom-inference""",0.004396,0.55719,0.00789
